In [1]:
import os
import sys
import glob
import warnings
import time
import numpy as np
from scipy import stats, signal
from scipy.io import loadmat
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import logging
import joblib

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
class Config:
    """Central configuration for the CZU-MHAD pipeline."""

    # Dataset path
    DATASET_ROOT = './CZU-MHAD/CZU-MHAD'

    # Subdirectories
    SKELETON_DIR = "skeleton_mat"
    INERTIAL_DIR = "sensor_mat"
    DEPTH_DIR = "depth_mat"

    SUBFOLDERS = ["depth_mat", "sensor_mat", "skeleton_mat"]

    # Output directory for saved features
    FEATURES_DIR = "features_new_10_loss"

    # Dataset properties
    NUM_ACTIONS = 22
    NUM_SUBJECTS = 5
    NUM_TRIALS = 5
    NUM_JOINTS = 20

    # Subject IDs (string-based in CZU-MHAD)
    ALL_SUBJECTS = ['cx', 'myj', 'zyh', 'cyy', 'qyh']
    TRAIN_SUBJECTS = ['cx', 'myj', 'zyh']
    TEST_SUBJECTS = ['cyy', 'qyh']

    # Inertial feature extraction parameters
    INERTIAL_WINDOW_SIZE = 50
    INERTIAL_WINDOW_OVERLAP = 0.5

    # Depth feature extraction parameters
    DEPTH_SAMPLE_FRAMES = 10

    # Random Forest parameters
    RF_N_ESTIMATORS = 300
    RF_MAX_DEPTH = None
    RF_MIN_SAMPLES_SPLIT = 2
    RF_MIN_SAMPLES_LEAF = 1
    RF_RANDOM_STATE = 42
    RF_N_JOBS = -1

    # Modalities
    USE_SKELETON = True
    USE_INERTIAL = True
    USE_DEPTH = True


In [3]:
# =============================================================================
# SECTION 2: DATA LOADING
# =============================================================================

def extract_numeric_array(mat_value):
    """
    Robustly extract a numeric numpy array from a .mat variable.
    Handles plain arrays, cell arrays, object arrays, nested structures.
    """
    # Already a plain numeric array
    if isinstance(mat_value, np.ndarray):
        # Object array (cell array from MATLAB) — drill into it
        if mat_value.dtype == object:
            # Try to find the numeric content inside
            # Common pattern: (1,1) object array wrapping the real data
            if mat_value.size == 1:
                inner = mat_value.flat[0]
                if isinstance(inner, np.ndarray):
                    return inner.astype(np.float64)
                else:
                    return np.array(inner, dtype=np.float64)
            # Multiple cells — try to stack them
            parts = []
            for item in mat_value.flat:
                if isinstance(item, np.ndarray) and item.size > 0:
                    parts.append(item.astype(np.float64))
            if parts:
                try:
                    return np.concatenate(parts, axis=0)
                except ValueError:
                    return parts[0]  # fallback to first element
            return None
        # Numeric dtype — use directly
        if np.issubdtype(mat_value.dtype, np.number):
            return mat_value.astype(np.float64)
        # Try forcing conversion
        try:
            return mat_value.astype(np.float64)
        except (ValueError, TypeError):
            return None
    return None


class CZUMHADLoader:
    """
    Loads the CZU-MHAD dataset from .mat files.

    File naming: {subject_id}_a{action_id}_t{trial_id}.mat
    Subjects: 'cx', 'myj', 'zyh', 'cyy', 'qyh'
    """

    def __init__(self, config):
        self.config = config
        self.root = config.DATASET_ROOT

    def load_all_data(self):
        """
        Load all modalities. Skeleton and sensor loaded into memory.
        Depth stores file paths only (lazy loading to avoid OOM).
        Only keeps samples present in ALL 3 modality directories.

        Returns: dict {basename: {action, subject, trial, skeleton, inertial, depth}}
        """
        sensor_dir = os.path.join(self.root, self.config.INERTIAL_DIR)
        all_files = [f for f in os.listdir(sensor_dir) if f.endswith(".mat")]
        subjects_found = sorted(list({f.split("_")[0] for f in all_files}))
        logger.info(f"Subjects found: {subjects_found}")
        logger.info(f"Total .mat files in {self.config.INERTIAL_DIR}: {len(all_files)}")

        # Step 1: Find samples that exist in ALL 3 modality folders
        valid_samples = []
        skipped = 0
        for file in sorted(all_files):
            base_name = file.replace(".mat", "")
            exists_in_all = all(
                os.path.exists(os.path.join(self.root, sub, base_name + ".mat"))
                for sub in self.config.SUBFOLDERS
            )
            if not exists_in_all:
                skipped += 1
                if skipped <= 5:
                    logger.warning(f"Skipping incomplete sample: {base_name}")
                continue

            parts = base_name.split("_")
            subj = parts[0]
            action = int(parts[1][1:])
            trial = int(parts[2][1:])
            valid_samples.append({
                'subject': subj, 'action': action, 'trial': trial,
                'basename': base_name,
            })

        if skipped > 0:
            logger.warning(f"Skipped {skipped} incomplete samples total")
        logger.info(f"Samples with all 3 modalities: {len(valid_samples)}")

        # Step 2: Debug — inspect first file from each modality
        if valid_samples:
            bn0 = valid_samples[0]['basename']
            for sub in self.config.SUBFOLDERS:
                fpath = os.path.join(self.root, sub, bn0 + ".mat")
                try:
                    mat = loadmat(fpath)
                    keys = [k for k in mat.keys() if not k.startswith("__")]
                    for k in keys:
                        v = mat[k]
                        logger.info(f"  DEBUG {sub}/{bn0}.mat key='{k}': "
                                    f"type={type(v).__name__}, "
                                    f"dtype={v.dtype if hasattr(v, 'dtype') else 'N/A'}, "
                                    f"shape={v.shape if hasattr(v, 'shape') else 'N/A'}")
                        if hasattr(v, 'dtype') and v.dtype == object and v.size <= 4:
                            for idx in range(min(v.size, 2)):
                                inner = v.flat[idx]
                                logger.info(f"    [flat {idx}]: type={type(inner).__name__}, "
                                            f"dtype={inner.dtype if hasattr(inner, 'dtype') else 'N/A'}, "
                                            f"shape={inner.shape if hasattr(inner, 'shape') else 'N/A'}")
                except Exception as e:
                    logger.info(f"  DEBUG error inspecting {sub}/{bn0}.mat: {e}")

        # Step 3: Load data
        samples = {}
        load_errors = 0
        first_errors = {}
        for sample_meta in valid_samples:
            bn = sample_meta['basename']
            try:
                # Load skeleton
                skel_path = os.path.join(self.root, "skeleton_mat", bn + ".mat")
                mat = loadmat(skel_path)
                key = [k for k in mat.keys() if not k.startswith("__")][0]
                skel_raw = extract_numeric_array(mat[key])
                if skel_raw is None:
                    raise ValueError(f"Cannot extract numeric array from skeleton key '{key}'")

                # Load sensor
                sensor_path = os.path.join(self.root, "sensor_mat", bn + ".mat")
                mat = loadmat(sensor_path)
                key = [k for k in mat.keys() if not k.startswith("__")][0]
                sensor_raw = extract_numeric_array(mat[key])
                if sensor_raw is None:
                    raise ValueError(f"Cannot extract numeric array from sensor key '{key}'")

                # Depth: filepath only
                depth_path = os.path.join(self.root, "depth_mat", bn + ".mat")

                samples[bn] = {
                    'action': sample_meta['action'],
                    'subject': sample_meta['subject'],
                    'trial': sample_meta['trial'],
                    'skeleton': skel_raw,
                    'inertial': sensor_raw,
                    'depth': depth_path,
                }
            except Exception as e:
                load_errors += 1
                err_msg = str(e)
                err_type = err_msg[:60]
                if err_type not in first_errors:
                    first_errors[err_type] = bn
                if load_errors <= 3:
                    logger.warning(f"Error loading {bn}: {e}")

        if load_errors > 3:
            logger.warning(f"Total loading errors: {load_errors}")
            logger.warning(f"Error types seen:")
            for err_type, first_bn in first_errors.items():
                logger.warning(f"  '{err_type}...' (first seen in: {first_bn})")

        logger.info(f"Successfully loaded {len(samples)} complete samples")

        if samples:
            first = next(iter(samples.values()))
            logger.info(f"  First sample skeleton: shape={first['skeleton'].shape}, dtype={first['skeleton'].dtype}")
            logger.info(f"  First sample sensor:   shape={first['inertial'].shape}, dtype={first['inertial'].dtype}")
            logger.info(f"  First sample depth:    filepath (lazy)")

        return samples


In [4]:
# =============================================================================
# SECTION 3: SKELETON FEATURE EXTRACTION
# =============================================================================

class SkeletonFeatureExtractor:
    """
    Extracts features from skeleton joint sequences.

    Inspired by:
    - VISTA paper: normalized joint coordinates relative to torso/hip,
      subset of most discriminative joints
    - MSDFE paper: covariance matrices capturing inter-joint correlations

    Feature categories:
    1. Static pose features (per-frame statistics)
    2. Dynamic motion features (temporal derivatives)
    3. Joint distance features (pairwise distances)
    4. Joint angle features (body part angles)
    5. Covariance features (inter-joint correlation matrix)
    """

    # UTD-MHAD Kinect joint indices (0-indexed):
    # 0: hip center, 1: spine, 2: shoulder center, 3: head,
    # 4: left shoulder, 5: left elbow, 6: left wrist, 7: left hand,
    # 8: right shoulder, 9: right elbow, 10: right wrist, 11: right hand,
    # 12: left hip, 13: left knee, 14: left ankle, 15: left foot,
    # 16: right hip, 17: right knee, 18: right ankle, 19: right foot

    HIP_CENTER = 0
    SPINE = 1
    SHOULDER_CENTER = 2
    HEAD = 3

    # Key joint groups for feature extraction
    UPPER_BODY = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
    LOWER_BODY = [0, 12, 13, 14, 15, 16, 17, 18, 19]
    HANDS = [7, 11]
    FEET = [15, 19]
    ENDPOINTS = [3, 7, 11, 15, 19]  # head, hands, feet

    # Joint pairs for distance features
    JOINT_PAIRS = [
        (7, 11),   # left hand - right hand
        (15, 19),  # left foot - right foot
        (7, 0),    # left hand - hip
        (11, 0),   # right hand - hip
        (7, 3),    # left hand - head
        (11, 3),   # right hand - head
        (15, 0),   # left foot - hip
        (19, 0),   # right foot - hip
        (6, 10),   # left wrist - right wrist
        (5, 9),    # left elbow - right elbow
        (4, 8),    # left shoulder - right shoulder
        (13, 17),  # left knee - right knee
    ]

    # Joint triplets for angle features (vertex is middle joint)
    ANGLE_TRIPLETS = [
        (4, 5, 6),    # left shoulder-elbow-wrist
        (8, 9, 10),   # right shoulder-elbow-wrist
        (12, 13, 14), # left hip-knee-ankle
        (16, 17, 18), # right hip-knee-ankle
        (4, 2, 8),    # left shoulder - shoulder center - right shoulder
        (5, 4, 2),    # left elbow - left shoulder - shoulder center
        (9, 8, 2),    # right elbow - right shoulder - shoulder center
        (0, 1, 2),    # hip - spine - shoulder center
    ]

    def __init__(self):
        pass

    def _normalize_skeleton(self, skel):
        """
        Normalize skeleton: translate to hip center, scale by torso length.
        Inspired by VISTA paper's torso-based normalization.

        Args:
            skel: (frames, 20, 3)
        Returns:
            normalized skeleton (frames, 20, 3)
        """
        # Translate: hip center as origin
        hip = skel[:, self.HIP_CENTER:self.HIP_CENTER+1, :]  # (frames, 1, 3)
        skel_norm = skel - hip

        # Scale: normalize by torso length (hip to shoulder center)
        torso_vec = skel_norm[:, self.SHOULDER_CENTER, :] - skel_norm[:, self.HIP_CENTER, :]
        torso_len = np.linalg.norm(torso_vec, axis=1, keepdims=True)  # (frames, 1)
        torso_len = np.clip(torso_len, 1e-6, None)
        skel_norm = skel_norm / torso_len[:, np.newaxis, :]

        return skel_norm

    def _compute_joint_distances(self, skel):
        """Compute pairwise joint distances over time."""
        distances = []
        for j1, j2 in self.JOINT_PAIRS:
            dist = np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1)
            distances.append(dist)
        return np.array(distances).T  # (frames, num_pairs)

    def _compute_joint_angles(self, skel):
        """Compute angles at joint triplets using dot product."""
        angles = []
        for j1, j2, j3 in self.ANGLE_TRIPLETS:
            v1 = skel[:, j1, :] - skel[:, j2, :]
            v2 = skel[:, j3, :] - skel[:, j2, :]
            cos_angle = np.sum(v1 * v2, axis=1) / (
                np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8
            )
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angle = np.arccos(cos_angle)
            angles.append(angle)
        return np.array(angles).T  # (frames, num_angles)

    def _compute_velocity(self, skel):
        """Compute first-order temporal derivatives (velocity)."""
        if skel.shape[0] < 2:
            return np.zeros_like(skel)
        vel = np.diff(skel, axis=0)
        vel = np.vstack([vel, vel[-1:]])  # pad to same length
        return vel

    def _compute_acceleration(self, skel):
        """Compute second-order temporal derivatives (acceleration)."""
        vel = self._compute_velocity(skel)
        if vel.shape[0] < 2:
            return np.zeros_like(vel)
        acc = np.diff(vel, axis=0)
        acc = np.vstack([acc, acc[-1:]])
        return acc

    def _temporal_statistics(self, signal_2d):
        """
        Compute temporal statistics over a 2D signal (frames, features).
        Inspired by VISTA paper's feature set.

        Returns: 1D feature vector with stats per feature column
        """
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s),
                np.std(s),
                np.sqrt(np.mean(s**2)),           # RMS
                stats.skew(s) if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s),            # range
                np.median(s),
                np.percentile(s, 25),             # Q1
                np.percentile(s, 75),             # Q3
            ])
        return np.array(features)

    def _covariance_features(self, skel):
        """
        Compute covariance matrix features (inspired by MSDFE paper).
        The covariance matrix captures inter-joint spatial correlations.

        Args:
            skel: (frames, 20, 3) normalized skeleton
        Returns:
            Upper triangular elements of the covariance matrix
        """
        # Flatten joints: (frames, 60)
        flat = skel.reshape(skel.shape[0], -1)

        # Select a subset of key joints to keep feature size manageable
        # Use endpoints + torso: head, hands, feet, hip, spine, shoulders
        key_joints = [0, 1, 2, 3, 7, 11, 15, 19]
        key_indices = []
        for j in key_joints:
            key_indices.extend([j*3, j*3+1, j*3+2])

        flat_key = flat[:, key_indices]  # (frames, 24)

        # Compute covariance matrix
        if flat_key.shape[0] < 2:
            cov_mat = np.zeros((flat_key.shape[1], flat_key.shape[1]))
        else:
            cov_mat = np.cov(flat_key.T)

        # Extract upper triangular elements (including diagonal)
        upper_tri = cov_mat[np.triu_indices(cov_mat.shape[0])]
        return upper_tri

    def extract(self, skel):
        """
        Extract full skeleton feature vector from a single sequence.

        Args:
            skel: raw skeleton data — CZU-MHAD shape is (frames, 100) or (frames, 20, 3)
        Returns:
            1D feature vector
        """
        if skel is None or skel.size == 0:
            return None

        # Handle CZU-MHAD raw (frames, 100) -> reshape to (frames, 20, 3)
        if skel.ndim == 2:
            n_frames, n_cols = skel.shape
            if n_cols >= 60:
                skel = skel[:, :60].reshape(n_frames, 20, 3)
            else:
                return None

        if skel.ndim != 3:
            return None

        # Ensure (frames, 20, 3) orientation
        if skel.shape[1] == 20 and skel.shape[2] == 3:
            pass  # already correct
        elif skel.shape[0] == 20 and skel.shape[2] == 3:
            skel = np.transpose(skel, (2, 0, 1))  # (3, 20, frames) -> wrong, try another
        elif skel.shape[2] == 20:
            skel = np.transpose(skel, (2, 1, 0))
        else:
            # Last resort: try the original UTD-MHAD transpose
            skel = np.transpose(skel, (2, 0, 1))

        # Handle potential NaN/Inf
        skel = np.nan_to_num(skel, nan=0.0, posinf=0.0, neginf=0.0)

        # 1. Normalize
        skel_norm = self._normalize_skeleton(skel)

        # 2. Joint positions - temporal statistics on all joint coords
        flat_pos = skel_norm.reshape(skel_norm.shape[0], -1)  # (frames, 60)
        # Use subset of key joints for position stats
        key_joint_ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 19]
        key_pos_idx = []
        for j in key_joint_ids:
            key_pos_idx.extend([j*3, j*3+1, j*3+2])
        pos_feats = self._temporal_statistics(flat_pos[:, key_pos_idx])

        # 3. Joint distances over time
        distances = self._compute_joint_distances(skel_norm)
        dist_feats = self._temporal_statistics(distances)

        # 4. Joint angles over time
        angles = self._compute_joint_angles(skel_norm)
        angle_feats = self._temporal_statistics(angles)

        # 5. Velocity features
        velocity = self._compute_velocity(skel_norm)
        vel_flat = velocity.reshape(velocity.shape[0], -1)
        # Key joints velocity
        vel_key = vel_flat[:, key_pos_idx]
        vel_feats = self._temporal_statistics(vel_key)

        # 6. Acceleration features
        accel = self._compute_acceleration(skel_norm)
        acc_flat = accel.reshape(accel.shape[0], -1)
        acc_key = acc_flat[:, key_pos_idx]
        acc_feats = self._temporal_statistics(acc_key)

        # 7. Covariance features (MSDFE-inspired)
        cov_feats = self._covariance_features(skel_norm)

        # 8. Global motion features
        # Center of mass trajectory
        com = np.mean(skel_norm, axis=1)  # (frames, 3)
        com_feats = self._temporal_statistics(com)

        # Total body movement (sum of all joint displacements)
        if skel_norm.shape[0] > 1:
            total_disp = np.sum(np.linalg.norm(np.diff(skel_norm, axis=0), axis=2), axis=1)
        else:
            total_disp = np.array([0.0])
        motion_energy = np.array([
            np.mean(total_disp), np.std(total_disp), np.max(total_disp),
            np.sum(total_disp)
        ])

        # Concatenate all skeleton features
        all_feats = np.concatenate([
            pos_feats,      # Position statistics
            dist_feats,     # Distance statistics
            angle_feats,    # Angle statistics
            vel_feats,      # Velocity statistics
            acc_feats,      # Acceleration statistics
            cov_feats,      # Covariance matrix features
            com_feats,      # Center of mass
            motion_energy,  # Global motion
        ])

        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)


In [5]:
# =============================================================================
# SECTION 4: INERTIAL FEATURE EXTRACTION
# =============================================================================

class InertialFeatureExtractor:
    """
    Extracts features from inertial sensor data (accelerometer + gyroscope).

    Directly inspired by the VISTA paper (Fiorini et al., 2022):
    - Time-domain features: mean, std, RMS, skewness, kurtosis, SMA, power
    - Additional: zero-crossing rate, peak count, signal energy
    - Frequency-domain: dominant frequency, spectral entropy, band energy

    The VISTA paper demonstrated these features on wrist + finger IMUs
    and showed 73-81% accuracy with individual sensors and higher with fusion.
    """

    def __init__(self, config):
        self.window_size = config.INERTIAL_WINDOW_SIZE
        self.overlap = config.INERTIAL_WINDOW_OVERLAP

    def _signal_magnitude_area(self, data):
        """
        Signal Magnitude Area (SMA) - used in VISTA paper.
        SMA = (1/N) * sum(|ax| + |ay| + |az|)
        """
        return np.mean(np.sum(np.abs(data), axis=1))

    def _signal_power(self, s):
        """Signal power = mean of squared values."""
        return np.mean(s**2)

    def _zero_crossing_rate(self, s):
        """Count zero crossings normalized by length."""
        s_centered = s - np.mean(s)
        return np.sum(np.abs(np.diff(np.sign(s_centered)))) / (2.0 * len(s))

    def _peak_count(self, s):
        """Count number of peaks in signal."""
        peaks, _ = signal.find_peaks(s)
        return len(peaks) / len(s)

    def _spectral_entropy(self, s, fs=50):
        """Compute spectral entropy of the signal."""
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        psd_norm = psd / (np.sum(psd) + 1e-12)
        psd_norm = psd_norm[psd_norm > 0]
        return -np.sum(psd_norm * np.log2(psd_norm + 1e-12))

    def _dominant_frequency(self, s, fs=50):
        """Find the dominant frequency component."""
        if len(s) < 4:
            return 0.0
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        return freqs[np.argmax(psd)]

    def _frequency_band_energy(self, s, fs=50, bands=[(0, 5), (5, 15), (15, 25)]):
        """Energy in different frequency bands."""
        if len(s) < 4:
            return [0.0] * len(bands)
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        energies = []
        for low, high in bands:
            mask = (freqs >= low) & (freqs < high)
            energies.append(np.sum(psd[mask]))
        return energies

    def _extract_channel_features(self, s):
        """
        Extract comprehensive features from a single channel.
        Based on VISTA paper's feature set + extensions.
        """
        features = []

        # Time-domain (VISTA paper features)
        features.append(np.mean(s))                          # Mean
        features.append(np.std(s))                           # Standard deviation
        features.append(np.sqrt(np.mean(s**2)))              # RMS
        features.append(stats.skew(s) if len(s) > 2 else 0) # Skewness
        features.append(stats.kurtosis(s) if len(s) > 3 else 0)  # Kurtosis
        features.append(self._signal_power(s))               # Power
        features.append(np.max(s) - np.min(s))               # Range
        features.append(np.median(s))                        # Median
        features.append(np.mean(np.abs(s)))                  # Mean absolute value
        features.append(np.percentile(s, 25))                # Q1
        features.append(np.percentile(s, 75))                # Q3
        features.append(np.percentile(s, 75) - np.percentile(s, 25))  # IQR
        features.append(self._zero_crossing_rate(s))         # Zero-crossing rate
        features.append(self._peak_count(s))                 # Peak count

        # Frequency-domain
        features.append(self._dominant_frequency(s))         # Dominant freq
        features.append(self._spectral_entropy(s))           # Spectral entropy
        features.extend(self._frequency_band_energy(s))      # Band energies

        return features

    def _extract_window_features(self, window):
        """
        Extract features from a single window of inertial data.

        Args:
            window: (window_size, 6) array [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        Returns:
            1D feature vector
        """
        features = []

        acc = window[:, :3]   # Accelerometer
        gyro = window[:, 3:]  # Gyroscope

        # Per-channel features for all 6 channels
        for ch in range(6):
            features.extend(self._extract_channel_features(window[:, ch]))

        # Accelerometer magnitude
        acc_mag = np.linalg.norm(acc, axis=1)
        features.extend(self._extract_channel_features(acc_mag))

        # Gyroscope magnitude
        gyro_mag = np.linalg.norm(gyro, axis=1)
        features.extend(self._extract_channel_features(gyro_mag))

        # Signal Magnitude Area (SMA) - key VISTA feature
        features.append(self._signal_magnitude_area(acc))
        features.append(self._signal_magnitude_area(gyro))

        # Cross-axis correlations (inspired by MSDFE covariance idea)
        for i in range(3):
            for j in range(i+1, 3):
                if len(acc[:, i]) > 1:
                    corr = np.corrcoef(acc[:, i], acc[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        for i in range(3):
            for j in range(i+1, 3):
                if len(gyro[:, i]) > 1:
                    corr = np.corrcoef(gyro[:, i], gyro[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        # Acc-Gyro cross-correlations
        for i in range(3):
            if len(acc[:, i]) > 1:
                corr = np.corrcoef(acc[:, i], gyro[:, i])[0, 1]
                features.append(corr if not np.isnan(corr) else 0.0)
            else:
                features.append(0.0)

        return features

    def extract(self, inertial_data):
        """
        Extract feature vector from a full inertial sequence.

        Strategy: segment into overlapping windows, extract features per window,
        then aggregate window features with statistics.

        Args:
            inertial_data: (samples, C) array — C can be 1 to 6 channels.
                           CZU-MHAD sensor is (10, 1).
        Returns:
            1D feature vector (always same length regardless of input size)
        """
        if inertial_data is None or inertial_data.size == 0:
            return None

        inertial_data = np.nan_to_num(inertial_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ensure 2D
        if inertial_data.ndim == 1:
            inertial_data = inertial_data.reshape(-1, 1)

        # If shape is (1, N) -> transpose to (N, 1)
        if inertial_data.ndim == 2 and inertial_data.shape[0] < inertial_data.shape[1]:
            inertial_data = inertial_data.T

        # Pad to 6 channels (the feature extractor always expects 6)
        if inertial_data.shape[1] < 6:
            pad_width = 6 - inertial_data.shape[1]
            inertial_data = np.hstack([
                inertial_data,
                np.zeros((inertial_data.shape[0], pad_width))
            ])

        # Apply low-pass Butterworth filter (VISTA paper uses 5Hz cutoff)
        try:
            b, a = signal.butter(4, 0.2, btype='low')
            for ch in range(inertial_data.shape[1]):
                if len(inertial_data[:, ch]) > 12:
                    inertial_data[:, ch] = signal.filtfilt(b, a, inertial_data[:, ch])
        except Exception:
            pass  # Skip filtering if it fails

        # Segment into windows
        n_samples = inertial_data.shape[0]
        step = int(self.window_size * (1 - self.overlap))
        step = max(step, 1)

        windows = []
        for start in range(0, n_samples - self.window_size + 1, step):
            windows.append(inertial_data[start:start + self.window_size])

        if not windows:
            # Sequence shorter than window -> use entire sequence as single window
            windows = [inertial_data]

        # Extract features from each window
        window_features = []
        for w in windows:
            wf = self._extract_window_features(w)
            window_features.append(wf)

        window_features = np.array(window_features)
        n_feat_per_window = window_features.shape[1]

        # ALWAYS aggregate with 4 statistics to ensure consistent output length
        # Even if there's only 1 window, we compute mean/std/min/max (std=0 for 1 window)
        aggregated = []
        for col in range(n_feat_per_window):
            col_data = window_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        return np.nan_to_num(np.array(aggregated), nan=0.0, posinf=0.0, neginf=0.0)


In [6]:
# =============================================================================
# SECTION 5: DEPTH FEATURE EXTRACTION
# =============================================================================

class DepthFeatureExtractor:
    """
    Extracts features from depth map sequences.

    Inspired by:
    - Siddiqi & Alrashdi (2022): Edge detection features from depth maps
      using Sobel operators, edge magnitude/direction histograms
    - MSDFE paper: Covariance-based statistical features from spatial regions

    Feature categories:
    1. Silhouette shape features (from thresholded depth)
    2. Edge-based features (Sobel gradients, inspired by Paper 3)
    3. Depth statistics (distribution of depth values)
    4. Temporal motion features (frame differences)
    5. HOG-like gradient histograms
    """

    def __init__(self, config):
        self.n_sample_frames = config.DEPTH_SAMPLE_FRAMES

    def _get_silhouette(self, depth_frame):
        """Extract binary silhouette from depth frame."""
        # Person is the closest non-zero depth
        valid = depth_frame[depth_frame > 0]
        if len(valid) == 0:
            return np.zeros_like(depth_frame, dtype=bool)
        threshold = np.percentile(valid, 50)
        return (depth_frame > 0) & (depth_frame < threshold)

    def _sobel_features(self, frame):
        """
        Compute Sobel edge features (inspired by Siddiqi paper).
        Edge magnitude and direction histograms.
        """
        # Sobel operators
        sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
        sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

        from scipy.ndimage import convolve
        gx = convolve(frame.astype(np.float64), sobel_x)
        gy = convolve(frame.astype(np.float64), sobel_y)

        # Edge magnitude
        magnitude = np.sqrt(gx**2 + gy**2)
        # Edge direction
        direction = np.arctan2(gy, gx + 1e-12)

        features = []

        # Magnitude statistics
        features.extend([
            np.mean(magnitude),
            np.std(magnitude),
            np.max(magnitude),
            np.sum(magnitude > np.mean(magnitude)),  # edge pixel count
        ])

        # Direction histogram (8 bins, like HOG)
        dir_hist, _ = np.histogram(direction[magnitude > np.mean(magnitude)],
                                    bins=8, range=(-np.pi, np.pi))
        if np.sum(dir_hist) > 0:
            dir_hist = dir_hist / (np.sum(dir_hist) + 1e-12)
        features.extend(dir_hist.tolist())

        # Gradient magnitude histogram
        mag_hist, _ = np.histogram(magnitude.ravel(), bins=8)
        if np.sum(mag_hist) > 0:
            mag_hist = mag_hist / (np.sum(mag_hist) + 1e-12)
        features.extend(mag_hist.tolist())

        return features

    def _shape_features(self, silhouette):
        """Compute shape descriptors from binary silhouette."""
        features = []

        area = np.sum(silhouette)
        features.append(area)

        if area == 0:
            return features + [0] * 7

        # Bounding box
        rows = np.any(silhouette, axis=1)
        cols = np.any(silhouette, axis=0)
        if np.any(rows) and np.any(cols):
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            height = rmax - rmin + 1
            width = cmax - cmin + 1
            features.append(height)
            features.append(width)
            features.append(height / (width + 1e-6))  # aspect ratio
            features.append(area / (height * width + 1e-6))  # extent
        else:
            features.extend([0, 0, 0, 0])

        # Centroid
        y_coords, x_coords = np.where(silhouette)
        if len(y_coords) > 0:
            features.append(np.mean(y_coords) / silhouette.shape[0])
            features.append(np.mean(x_coords) / silhouette.shape[1])
            features.append(np.std(y_coords) / (silhouette.shape[0] + 1e-6))
        else:
            features.extend([0, 0, 0])

        return features

    def _depth_distribution_features(self, depth_frame):
        """Statistical features from depth value distribution."""
        valid = depth_frame[depth_frame > 0].ravel()
        if len(valid) == 0:
            return [0] * 8

        features = [
            np.mean(valid),
            np.std(valid),
            np.median(valid),
            np.min(valid),
            np.max(valid),
            stats.skew(valid) if len(valid) > 2 else 0,
            stats.kurtosis(valid) if len(valid) > 3 else 0,
            np.percentile(valid, 75) - np.percentile(valid, 25),  # IQR
        ]
        return features

    def _extract_frame_features(self, frame):
        """Extract features from a single depth frame."""
        features = []

        # Resize/downsample for efficiency
        h, w = frame.shape
        scale = 4
        small = frame[::scale, ::scale]

        # Silhouette features
        sil = self._get_silhouette(small)
        features.extend(self._shape_features(sil))

        # Edge features (Siddiqi-inspired)
        features.extend(self._sobel_features(small))

        # Depth distribution
        features.extend(self._depth_distribution_features(small))

        return features

    def _load_depth_from_path(self, filepath):
        """Load depth data from .mat file path (lazy loading to save memory)."""
        try:
            mat = loadmat(filepath)
            key = [k for k in mat.keys() if not k.startswith("__")][0]
            return extract_numeric_array(mat[key])
        except Exception as e:
            logger.debug(f"Error loading depth {filepath}: {e}")
            return None

    def extract(self, depth_data):
        """
        Extract feature vector from a depth sequence.

        Args:
            depth_data: filepath string (lazy load) OR (frames, H, W) array
        Returns:
            1D feature vector (always same length: n_feats_per_frame * 6)
        """
        # Support lazy loading from file path
        if isinstance(depth_data, str):
            depth_data = self._load_depth_from_path(depth_data)

        if depth_data is None or depth_data.size == 0:
            return None

        depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ensure shape is (frames, height, width)
        if depth_data.ndim == 3:
            if depth_data.shape[0] > depth_data.shape[2]:
                # Likely (height, width, frames) -> transpose
                depth_data = np.transpose(depth_data, (2, 0, 1))
        else:
            return None

        n_frames = depth_data.shape[0]

        # ALWAYS sample exactly n_sample_frames for consistent output size
        if n_frames == 0:
            return None
        if n_frames >= self.n_sample_frames:
            frame_indices = np.linspace(0, n_frames - 1, self.n_sample_frames, dtype=int)
        else:
            # Repeat frames to reach n_sample_frames
            frame_indices = np.array([i % n_frames for i in range(self.n_sample_frames)])

        # Extract per-frame features
        frame_features = []
        for idx in frame_indices:
            ff = self._extract_frame_features(depth_data[idx])
            frame_features.append(ff)

        frame_features = np.array(frame_features)

        # Temporal features from frame differences (always n_sample_frames - 1 diffs)
        diffs = np.diff(frame_features, axis=0)
        temporal_feats = []
        for col in range(diffs.shape[1]):
            temporal_feats.extend([
                np.mean(diffs[:, col]),
                np.std(diffs[:, col]),
            ])

        # Aggregate frame features
        aggregated = []
        for col in range(frame_features.shape[1]):
            col_data = frame_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        all_feats = np.concatenate([aggregated, temporal_feats])
        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)


In [7]:
# =============================================================================
# SECTION 6: MULTIMODAL FUSION & CLASSIFICATION PIPELINE
# =============================================================================

class MultimodalHARPipeline:
    """
    Complete pipeline: load data, extract features, fuse, classify.
    """

    def __init__(self, config):
        self.config = config
        self.loader = CZUMHADLoader(config)
        self.skel_extractor = SkeletonFeatureExtractor()
        self.iner_extractor = InertialFeatureExtractor(config)
        self.depth_extractor = DepthFeatureExtractor(config)
        self.scaler = StandardScaler()


    def load_all_data(self):
        """Load all modalities via the unified loader."""
        return self.loader.load_all_data()


    def extract_features(self, samples):
        """
        Extract features from all samples across all modalities.
        Skips entire sample if ANY modality extraction fails.
        Validates consistent feature dimensions across all samples.

        Returns:
            features_dict: {key: {features, label, subject, modality_features, ...}}
        """
        features_dict = {}
        total = len(samples)
        skipped = 0
        expected_dims = None  # will be set from first successful sample

        for i, (key, sample) in enumerate(samples.items()):
            if (i + 1) % 100 == 0 or i == 0:
                logger.info(f"Extracting features: {i+1}/{total}")

            # Extract each modality — ALL must succeed
            skel_feats = self.skel_extractor.extract(sample['skeleton'])
            if skel_feats is None:
                skipped += 1
                continue

            iner_feats = self.iner_extractor.extract(sample['inertial'])
            if iner_feats is None:
                skipped += 1
                continue

            depth_feats = self.depth_extractor.extract(sample['depth'])
            if depth_feats is None:
                skipped += 1
                continue

            # Check consistent dimensions
            dims = (len(skel_feats), len(iner_feats), len(depth_feats))
            if expected_dims is None:
                expected_dims = dims
                logger.info(f"  Feature dimensions: skeleton={dims[0]}, "
                            f"sensor={dims[1]}, depth={dims[2]}, "
                            f"total={sum(dims)}")
            elif dims != expected_dims:
                logger.warning(f"  Inconsistent dims for {key}: {dims} vs expected {expected_dims}, skipping")
                skipped += 1
                continue

            concatenated = np.concatenate([skel_feats, iner_feats, depth_feats])

            features_dict[key] = {
                'features': concatenated,
                'label': sample['action'],
                'subject': sample['subject'],
                'modalities': ['skeleton', 'inertial', 'depth'],
                'modality_sizes': {
                    'skeleton': len(skel_feats),
                    'inertial': len(iner_feats),
                    'depth': len(depth_feats),
                },
                'modality_features': {
                    'skeleton': skel_feats,
                    'inertial': iner_feats,
                    'depth': depth_feats,
                },
            }

        if skipped > 0:
            logger.warning(f"Skipped {skipped} samples due to extraction failures")
        logger.info(f"Extracted features for {len(features_dict)}/{total} samples")

        if features_dict:
            sample_entry = next(iter(features_dict.values()))
            logger.info(f"Total feature vector dimension: {len(sample_entry['features'])}")
            for mod, size in sample_entry['modality_sizes'].items():
                logger.info(f"  {mod}: {size} features")

        return features_dict


    def save_features(self, features_dict):
        """
        Save extracted features for later use.

        Saves:
          - X_feat.pkl: list of dicts with 'depth_feat', 'sensor_feat', 'skeleton_feat'
          - y.npy: encoded action labels
          - subjects.npy: subject IDs per sample
          - label_encoder.pkl: fitted LabelEncoder
        """
        out_dir = self.config.FEATURES_DIR
        os.makedirs(out_dir, exist_ok=True)

        keys = sorted(features_dict.keys())

        X_feat = []
        y_raw = []
        subjects = []

        for k in keys:
            entry = features_dict[k]
            mod_feats = entry.get('modality_features', {})

            sample_dict = {
                'skeleton_feat': mod_feats.get('skeleton', np.array([])),
                'sensor_feat':   mod_feats.get('inertial', np.array([])),
                'depth_feat':    mod_feats.get('depth', np.array([])),
            }
            X_feat.append(sample_dict)
            y_raw.append(entry['label'])
            subjects.append(entry['subject'])

        y_raw = np.array(y_raw)
        subjects = np.array(subjects)

        le = LabelEncoder()
        y = le.fit_transform(y_raw)

        joblib.dump(X_feat, os.path.join(out_dir, 'X_feat.pkl'))
        np.save(os.path.join(out_dir, 'y.npy'), y)
        np.save(os.path.join(out_dir, 'subjects.npy'), subjects)
        joblib.dump(le, os.path.join(out_dir, 'label_encoder.pkl'))

        logger.info(f"Features saved to '{out_dir}/':")
        logger.info(f"  X_feat.pkl:        {len(X_feat)} samples")
        first = X_feat[0]
        for mod_key in ['skeleton_feat', 'sensor_feat', 'depth_feat']:
            dim = first[mod_key].shape[0] if first[mod_key].size > 0 else 0
            logger.info(f"    {mod_key}: {dim} features")
        total_dim = sum(first[m].shape[0] for m in first if first[m].size > 0)
        logger.info(f"    TOTAL: {total_dim} features")
        logger.info(f"  y.npy:             {y.shape} (encoded, {len(le.classes_)} classes)")
        logger.info(f"  subjects.npy:      {subjects.shape} "
                     f"(subjects: {sorted(np.unique(subjects).tolist())})")
        logger.info(f"  label_encoder.pkl: classes = {le.classes_.tolist()}")


    def split_train_test(self, features_dict):
        """Split into train/test using subject-based protocol."""
        X_train, y_train = [], []
        X_test, y_test = [], []

        for key, entry in features_dict.items():
            if entry['subject'] in self.config.TRAIN_SUBJECTS:
                X_train.append(entry['features'])
                y_train.append(entry['label'])
            elif entry['subject'] in self.config.TEST_SUBJECTS:
                X_test.append(entry['features'])
                y_test.append(entry['label'])

        X_train = np.array(X_train, dtype=np.float64)
        y_train = np.array(y_train)
        X_test = np.array(X_test, dtype=np.float64)
        y_test = np.array(y_test)

        logger.info(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
        logger.info(f"Feature dimension: {X_train.shape[1]}")
        logger.info(f"Number of classes: {len(np.unique(y_train))}")

        return X_train, y_train, X_test, y_test


    def train_and_evaluate(self, X_train, y_train, X_test, y_test):
        """Train Random Forest and evaluate."""
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

        rf = RandomForestClassifier(
            n_estimators=self.config.RF_N_ESTIMATORS,
            max_depth=self.config.RF_MAX_DEPTH,
            min_samples_split=self.config.RF_MIN_SAMPLES_SPLIT,
            min_samples_leaf=self.config.RF_MIN_SAMPLES_LEAF,
            random_state=self.config.RF_RANDOM_STATE,
            n_jobs=self.config.RF_N_JOBS,
            class_weight='balanced',
        )

        logger.info("Training Random Forest classifier...")
        start_time = time.time()
        rf.fit(X_train_scaled, y_train)
        train_time = time.time() - start_time
        logger.info(f"Training completed in {train_time:.2f} seconds")

        y_pred = rf.predict(X_test_scaled)

        accuracy = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')

        logger.info(f"\n{'='*60}")
        logger.info(f"RESULTS")
        logger.info(f"{'='*60}")
        logger.info(f"Overall Accuracy: {accuracy*100:.2f}%")
        logger.info(f"Macro F1-Score:   {f1_macro*100:.2f}%")
        logger.info(f"Weighted F1-Score: {f1_weighted*100:.2f}%")

        action_names = [f"Action {i}" for i in range(1, self.config.NUM_ACTIONS + 1)]
        unique_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
        target_names = [action_names[l-1] for l in unique_labels]

        report = classification_report(y_test, y_pred, labels=unique_labels,
                                        target_names=target_names, digits=3)
        logger.info(f"\nClassification Report:\n{report}")

        logger.info("Running 5-fold cross-validation on training set...")
        cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring='accuracy')
        logger.info(f"CV Accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")

        importances = rf.feature_importances_
        top_k = 20
        top_indices = np.argsort(importances)[-top_k:][::-1]
        logger.info(f"\nTop {top_k} most important features (by index):")
        for idx in top_indices:
            logger.info(f"  Feature {idx}: importance = {importances[idx]:.4f}")

        return {
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'f1_weighted': f1_weighted,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'train_time': train_time,
            'model': rf,
            'y_pred': y_pred,
        }


    def run(self):
        """Execute the full pipeline."""
        logger.info("="*60)
        logger.info("MULTIMODAL HAR PIPELINE FOR CZU-MHAD")
        logger.info("="*60)
        logger.info(f"Modalities: Skeleton={self.config.USE_SKELETON}, "
                     f"Inertial={self.config.USE_INERTIAL}, "
                     f"Depth={self.config.USE_DEPTH}")
        logger.info(f"Train subjects: {self.config.TRAIN_SUBJECTS}")
        logger.info(f"Test subjects:  {self.config.TEST_SUBJECTS}")

        logger.info("\n--- Step 1: Loading data ---")
        samples = self.load_all_data()

        if not samples:
            logger.error("No data loaded! Check the dataset path.")
            logger.info(f"Expected path: {self.config.DATASET_ROOT}")
            return None
        
        # --- Zero out random 10% of raw skeleton and inertial data ---
        rng = np.random.RandomState(42)
        for sample in samples.values():
            for mod in ('skeleton', 'inertial'):
                arr = sample[mod]
                if arr is not None and arr.size > 0:
                    mask = rng.random(arr.shape) < 0.10
                    arr[mask] = 0.0
                    
        logger.info("Zeroed out 10% of raw skeleton and inertial data")

        logger.info("\n--- Step 2: Extracting features ---")
        features_dict = self.extract_features(samples)

        if not features_dict:
            logger.error("No features extracted!")
            return None

        logger.info("\n--- Step 2b: Saving features ---")
        self.save_features(features_dict)

        logger.info("\n--- Step 3: Train/test split ---")
        X_train, y_train, X_test, y_test = self.split_train_test(features_dict)

        if X_train.shape[0] == 0 or X_test.shape[0] == 0:
            logger.error("Empty train or test set!")
            return None

        logger.info("\n--- Step 4: Training and evaluation ---")
        results = self.train_and_evaluate(X_train, y_train, X_test, y_test)

        return results


In [8]:
# =============================================================================
# SECTION 7: DEMO MODE (generates synthetic data for testing)
# =============================================================================

def create_synthetic_dataset(root_dir, n_actions=22, subjects=None, n_trials=5):
    """Create synthetic CZU-MHAD-like dataset for testing."""
    if subjects is None:
        subjects = ['cx', 'myj', 'zyh', 'cyy', 'qyh']

    logger.info("Creating synthetic dataset for demonstration...")

    for modality, subdir in [("skeleton", "skeleton_mat"), ("sensor", "sensor_mat"),
                              ("depth", "depth_mat")]:
        mod_dir = os.path.join(root_dir, subdir)
        os.makedirs(mod_dir, exist_ok=True)

        for a in range(1, n_actions + 1):
            for s in subjects:
                for t in range(1, n_trials + 1):
                    fname = f"{s}_a{a}_t{t}.mat"
                    fpath = os.path.join(mod_dir, fname)

                    if os.path.exists(fpath):
                        continue

                    from scipy.io import savemat

                    if modality == "skeleton":
                        # (59, 100)
                        n_frames = np.random.randint(30, 80)
                        base = np.random.randn(n_frames, 100) * 0.1
                        base[:, 0] += np.sin(np.linspace(0, a * np.pi, n_frames))
                        savemat(fpath, {'skeleton': base})

                    elif modality == "sensor":
                        # (10, 1)
                        data = np.random.randn(10, 1) * 0.5
                        data[:, 0] += np.sin(np.linspace(0, a * 0.5, 10))
                        savemat(fpath, {'sensor': data})

                    elif modality == "depth":
                        # (60, 106, 128) - small for synthetic demo
                        h, w = 106, 128
                        n_depth_frames = 20
                        depth = np.zeros((n_depth_frames, h, w))
                        for fr in range(n_depth_frames):
                            cy = h // 2 + int(5 * np.sin(a * fr / 10.0))
                            cx_pos = w // 2 + int(5 * np.cos(a * fr / 10.0))
                            Y, X = np.ogrid[:h, :w]
                            mask = (Y - cy)**2 + (X - cx_pos)**2 < (10 + a)**2
                            depth[fr][mask] = 1000 + a * 50 + np.random.rand() * 100
                        savemat(fpath, {'depth': depth})

    logger.info(f"Synthetic dataset created at {root_dir}")


# =============================================================================
# SECTION 8: MAIN EXECUTION
# =============================================================================

def main():
    """Main entry point."""
    config = Config()

    if not os.path.isdir(config.DATASET_ROOT):
        logger.info(f"Dataset not found at '{config.DATASET_ROOT}'")
        logger.info("Creating synthetic dataset for demonstration...")
        os.makedirs(config.DATASET_ROOT, exist_ok=True)
        create_synthetic_dataset(config.DATASET_ROOT)

    pipeline = MultimodalHARPipeline(config)
    results = pipeline.run()

    if results:
        logger.info(f"\n{'='*60}")
        logger.info("FINAL SUMMARY")
        logger.info(f"{'='*60}")
        logger.info(f"Accuracy:         {results['accuracy']*100:.2f}%")
        logger.info(f"Macro F1:         {results['f1_macro']*100:.2f}%")
        logger.info(f"Weighted F1:      {results['f1_weighted']*100:.2f}%")
        logger.info(f"CV Accuracy:      {results['cv_mean']*100:.2f}% "
                     f"(+/- {results['cv_std']*100:.2f}%)")
        logger.info(f"Training Time:    {results['train_time']:.2f}s")


if __name__ == "__main__":
    main()


2026-03-03 12:23:34,672 - INFO - ============================================================
2026-03-03 12:23:34,672 - INFO - MULTIMODAL HAR PIPELINE FOR CZU-MHAD
2026-03-03 12:23:34,672 - INFO - ============================================================
2026-03-03 12:23:34,672 - INFO - Modalities: Skeleton=True, Inertial=True, Depth=True
2026-03-03 12:23:34,672 - INFO - Train subjects: ['cx', 'myj', 'zyh']
2026-03-03 12:23:34,672 - INFO - Test subjects:  ['cyy', 'qyh']
2026-03-03 12:23:34,673 - INFO - 
--- Step 1: Loading data ---
2026-03-03 12:23:34,673 - INFO - Subjects found: ['cx', 'cyy', 'myj', 'qyh', 'zyh']
2026-03-03 12:23:34,673 - INFO - Total .mat files in sensor_mat: 1165
2026-03-03 12:23:34,682 - INFO - Samples with all 3 modalities: 1165
2026-03-03 12:23:34,737 - INFO -   DEBUG depth_mat/cx_a10_t1.mat key='depth': type=ndarray, dtype=uint8, shape=(132, 424, 512)
2026-03-03 12:23:34,742 - INFO -   DEBUG sensor_mat/cx_a10_t1.mat key='sensor': type=ndarray, dtype=object, s